# EXP-023 겨냥 지표: 근거 번호 지시를 보강한 v10

EXP-023 판정(보류) 뒤에 쓴다. 겨냥 지표 A(코드 집계)를 기준(v8)·v9·v10에서 같은 코드로 내고, B(표본 판독)의 표본을 만든다. 새 API 호출이 없고 저장된 기록만 읽는다. 질의 원문과 근거·답변 원문은 출력하지 않는다.

In [1]:
import json
import random
import re
from collections import Counter
from pathlib import Path

import pandas as pd

from rag.conditions import committee_base
from rag.gen_eval.checks import CITE, doc_date, metadata_mismatches, sentences
from rag.gen_eval.report import per_query
from rag.index import load_cfg

In [2]:
ROOT = Path.cwd().parent
PREFIX = "hybrid-cond-filter-law-ms256-k20-ce"
JUDGE = "deepseek_go_mixed-j_v13"
RUNS = {"v8": "v8-dev-full-spk-marks", "v9": "v9-dev-full-spk-marks", "v10": "v10-dev-full-spk-marks"}
JUDGE_SUFFIX = {"v8": "-s34", "v9": "", "v10": ""}
SEED = 20260929
N_SAMPLE = 30
BASE_KEY = ROOT / "_workspace/exp022_b_key.json"
SAMPLE = ROOT / "_workspace/exp023_b_sample.jsonl"
SAMPLE_KEY = ROOT / "_workspace/exp023_b_key.json"
LABELS = ROOT / "_workspace/exp023_b_labels.jsonl"
cfg = load_cfg([])
FILLERS = list(cfg.gen_eval.filler_phrases)

In [3]:
def read(path):
    with open(path, encoding="utf-8") as f:
        return {r["qid"]: r for r in map(json.loads, f)}


def cited(g):
    """답변이 인용한 근거 번호."""
    return {int(m) for m in CITE.findall(g["answer"]) if 1 <= int(m) <= len(g["contexts"])}


def year(c):
    return (doc_date(c["date"]) or str(c["date"]))[:4]


def names_meeting(s, nums, ctx):
    """문장 안에 인용 근거의 연도(NNNN년)나 위원회 이름이 있는가."""
    flat = s.replace(" ", "")
    return any(f"{year(ctx[i - 1])}년" in flat or committee_base(ctx[i - 1]["committee_name"]).replace(" ", "") in flat for i in nums)

## 데이터 로딩

In [4]:
gens = {k: read(ROOT / f"data/gen_eval/runs/{PREFIX}-{v}.jsonl") for k, v in RUNS.items()}
judged = {k: read(ROOT / f"data/gen_eval/judge/{PREFIX}-{v}-{JUDGE}{JUDGE_SUFFIX[k]}.jsonl") for k, v in RUNS.items()}
qids = list(gens["v8"])
print({k: (len(gens[k]), len(judged[k])) for k in RUNS})

{'v8': (330, 330), 'v9': (330, 330), 'v10': (330, 330)}


## A. 코드 집계

인용이 전혀 없는 답변이 이 실험의 추가 겨냥이다. 나머지는 EXP-022와 같은 코드다.

In [5]:
def aggregate(k):
    n = Counter()
    for q, r in gens[k].items():
        ctx = r["contexts"]
        nums = cited(r)
        full = judged[k][q]["refusal"] == "full"
        n["인용이 전혀 없는 답변(전체 거부 제외)"] += not nums and not full
        n["2개 이상 문서 인용"] += len({ctx[i - 1]["doc_id"] for i in nums}) >= 2
        n["2개 이상 연도 인용"] += len({year(ctx[i - 1]) for i in nums}) >= 2
        n["2개 이상 위원회 인용"] += len({committee_base(ctx[i - 1]["committee_name"]) for i in nums}) >= 2
        n["1위 근거만 인용"] += nums == {1}
        n["2위 이하도 인용"] += bool(nums - {1})
        multi = len({ctx[i - 1]["doc_id"] for i in nums}) >= 2
        mixed = False
        for s in sentences(r["answer"]):
            ns = {int(m) for m in CITE.findall(s) if 1 <= int(m) <= len(ctx)}
            if not ns:
                continue
            mixed |= len({year(ctx[i - 1]) for i in ns}) >= 2
            n["인용 문장"] += 1
            n["회의를 밝힌 인용 문장"] += names_meeting(s, ns, ctx)
            if multi:
                n["여러 문서 답변의 인용 문장"] += 1
                n["여러 문서 답변에서 회의를 밝힌 문장"] += names_meeting(s, ns, ctx)
        n["한 문장에 2개 이상 연도 인용"] += mixed
    return n


pd.DataFrame({k: aggregate(k) for k in RUNS})

,v8,v9,v10
인용이 전혀 없는 답변(전체 거부 제외),0,11,1
2개 이상 문서 인용,85,96,91
2개 이상 연도 인용,48,58,56
2개 이상 위원회 인용,44,48,46
1위 근거만 인용,237,216,230
2위 이하도 인용,87,99,94
인용 문장,726,722,742
회의를 밝힌 인용 문장,14,459,471
한 문장에 2개 이상 연도 인용,2,0,1
여러 문서 답변의 인용 문장,319,350,338


## 회의 일자·위원회 불일치(S37 함수로 다시 셈)

저장값은 실행마다 규칙이 달라 세 실행을 같은 함수로 다시 센다. S37에서 관찰로 둔 두 경우(연도를 채운 날짜, 본문의 다른 위원회 이름)가 나오는지도 값만 본다.

In [6]:
names = {committee_base(c["committee_name"]) for g in gens.values() for r in g.values() for c in r["contexts"]}
for k in RUNS:
    found = [(q, m["field"], m["value"]) for q, r in gens[k].items() if judged[k][q]["refusal"] != "full" for m in metadata_mismatches(r["answer"], r["contexts"], names)]
    print(k, "답변", len({q for q, _, _ in found}), "건", len(found), Counter(f for _, f, _ in found))

v8 답변 0 건 0 Counter()
v9 답변 0 건 0 Counter()
v10 답변 0 건 0 Counter()


## 답변 형식과 관련성

불릿 답변 수와 관련성 차이의 나뉨을 본다.

In [7]:
BULLET = re.compile(r"(?m)^[ \t]*(?:[-*•·]|[0-9]+[.)])[ \t]")
print({k: sum(bool(BULLET.search(r["answer"])) for r in gens[k].values()) for k in RUNS})
pq = {k: per_query(list(judged[k].values())) for k in RUNS}
for k in ("v9", "v10"):
    d = pq[k]["relevance"] - pq["v8"]["relevance"]
    print(k, "관련성 차이 합", round(d.sum(), 2), "오름", int((d > 0).sum()), "내림", int((d < 0).sum()), "같음", int((d == 0).sum()))
nocite = [q for q in qids if not cited(gens["v10"][q]) and judged["v10"][q]["refusal"] != "full"]
print("v10 인용 없는 답변의 관련성", Counter((judged["v10"][q].get("relevance") or {}).get("score") for q in nocite))

{'v8': 35, 'v9': 43, 'v10': 44}
v9 관련성 차이 합 -3.5 오름 19 내림 26 같음 278
v10 관련성 차이 합 -1.0 오름 25 내림 26 같음 271
v10 인용 없는 답변의 관련성 Counter({1.0: 1})


## B. 판정 뒤 표본 판독: 표본 만들기

2개 이상 문서를 인용한 v10 답변 30개와 EXP-022의 기준 표본 30개(같은 답변)를 섞는다. 판독자가 달라도 같은 기준으로 비교하려고 기준 표본을 다시 판독한다. 실행을 가리고 질의 원문은 넣지 않는다. 표본 파일은 `_workspace/`에 두고 커밋하지 않는다.

In [8]:
rng = random.Random(SEED)
pool = sorted(q for q in qids if len({gens["v10"][q]["contexts"][i - 1]["doc_id"] for i in cited(gens["v10"][q])}) >= 2)
base_q = [v["qid"] for v in json.loads(BASE_KEY.read_text(encoding="utf-8")).values() if v["run"] == "base"]
picked = [("v10", q) for q in rng.sample(pool, N_SAMPLE)] + [("v8", q) for q in base_q]
rng.shuffle(picked)
if SAMPLE.exists():
    raise SystemExit(f"{SAMPLE}가 이미 있다. 판독 뒤 덮어쓰면 짝이 깨진다")
key = {}
with open(SAMPLE, "w", encoding="utf-8", newline="\n") as f:
    for i, (k, q) in enumerate(picked):
        g = gens[k][q]
        evid = [{"n": n, "date": g["contexts"][n - 1]["date"], "committee": g["contexts"][n - 1]["committee_name"], "text": g["contexts"][n - 1]["text"]} for n in sorted(cited(g))]
        key[i] = {"run": k, "qid": q}
        f.write(json.dumps({"id": i, "answer": g["answer"], "evidence": evid}, ensure_ascii=False) + "\n")
SAMPLE_KEY.write_text(json.dumps(key, ensure_ascii=False), encoding="utf-8")
print(len(picked), Counter(v["run"] for v in key.values()), "v10 후보 집단", len(pool))

60 Counter({'v10': 30, 'v8': 30}) v10 후보 집단 91


## B. 판정 뒤 표본 판독: 결과

서브에이전트 1회가 실행을 가리고 판독했다. 라벨을 실행과 대조해 묶음 비율과 Wilson 95% 구간을 낸다. 기준 표본 30개는 EXP-022와 같은 답변이라 두 판독의 일치도도 본다.

In [9]:
key = json.loads(SAMPLE_KEY.read_text(encoding="utf-8"))
with open(LABELS, encoding="utf-8") as f:
    labels = [json.loads(line) for line in f]
tab = pd.crosstab(pd.Series([key[str(r["id"])]["run"] for r in labels], name="실행"), pd.Series([r["label"] for r in labels], name="라벨"))
print(tab)


def wilson(k, n, z=1.96):
    p = k / n
    mid = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * ((p * (1 - p) + z * z / (4 * n)) / n) ** 0.5 / (1 + z * z / n)
    return round(p, 3), round(mid - half, 3), round(mid + half, 3)


print({k: wilson(int(tab.loc[k, "묶음"]), int(tab.loc[k].sum())) for k in ("v8", "v10")})

라벨   같은 사안  나눔  묶음
실행                
v10     14  16   0
v8      15   7   8
{'v8': (0.267, 0.142, 0.444), 'v10': (0.0, -0.0, 0.114)}


In [10]:
# EXP-022 판독과 이번 판독이 같은 기준 답변 30개에 붙인 라벨을 대조한다
old_key = json.loads(BASE_KEY.read_text(encoding="utf-8"))
with open(ROOT / "_workspace/exp022_b_labels.jsonl", encoding="utf-8") as f:
    old = {old_key[str(r["id"])]["qid"]: r["label"] for r in map(json.loads, f) if old_key[str(r["id"])]["run"] == "base"}
new = {key[str(r["id"])]["qid"]: r["label"] for r in labels if key[str(r["id"])]["run"] == "v8"}
print("일치", sum(old[q] == new[q] for q in old), "/", len(old))
print(pd.crosstab(pd.Series([old[q] for q in old], name="EXP-022 판독"), pd.Series([new[q] for q in old], name="이번 판독")))

일치 28 / 30
이번 판독       같은 사안  나눔  묶음
EXP-022 판독               
같은 사안          15   0   1
나눔              0   7   1
묶음              0   0   6


## 정리

- 인용이 전혀 없는 답변은 v9 11 → v10 1이고, 회의를 밝힌 인용 문장은 471/742로 유지됐다. 회의 정보 불일치는 S37 함수로 세 실행 모두 0이다.
- 표본 묶음 8/30 → 0/30이고, 같은 기준 답변의 재판독은 28/30 일치한다.
- 사용자는 v10을 예외 채택했다(D-76).